# Autonomous Driving Perception System - Complete Self-Contained Demo

## Multi-Sensor Fusion for 3D Object Detection and Path Planning

**This notebook is 100% self-contained!**

Just download this file and click **Run All** - no setup required!

### What this demonstrates:
1. Automatic data generation (simulated LiDAR, Radar, Camera data)
2. 3D object detection using PointPillars
3. Multi-object tracking with Kalman filter
4. Path planning with A* algorithm
5. Trajectory optimization
6. Complete visualization

**No external files needed - everything is embedded in this notebook!**

In [ ]:
# Install dependencies (runs automatically)
import sys
import subprocess

print('Checking required packages...')
required = ['torch', 'numpy', 'matplotlib', 'scipy']
missing = []

for pkg in required:
    try:
        __import__(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        missing.append(pkg)
        print(f'  ✗ {pkg} - will install')

if missing:
    print(f'\nInstalling {len(missing)} missing package(s)...')
    for pkg in missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print('✓ Installation complete!\n')
else:
    print('✓ All packages already installed!\n')

# Import standard libraries
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for faster execution
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
import warnings
import heapq
from typing import List, Tuple, Dict, Optional

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 8)

print('='*80)
print('AUTONOMOUS DRIVING PERCEPTION SYSTEM')
print('='*80)
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')
print('='*80)

## Step 1: Data Generation

Generate simulated autonomous driving data (LiDAR, Radar, Camera, 3D boxes)

In [ ]:
def generate_autonomous_driving_data(num_samples=100, seed=42):
    """
    Generate simulated multi-sensor autonomous driving data.
    
    Returns list of samples, each containing:
    - lidar_points: (N, 4) array of x, y, z, intensity
    - radar_points: (M, 5) array of x, y, z, vx, vy
    - camera_image: (H, W, 3) RGB image
    - gt_boxes_3d: (K, 9) array of x, y, z, w, l, h, yaw, class, track_id
    """
    np.random.seed(seed)
    dataset = []
    
    for idx in range(num_samples):
        np.random.seed(seed + idx)
        
        # Generate LiDAR point cloud
        num_points = np.random.randint(10000, 30000)
        lidar_points = np.random.randn(num_points, 4).astype(np.float32)
        lidar_points[:, 0] = lidar_points[:, 0] * 20 + 30  # x: 10-50m
        lidar_points[:, 1] = lidar_points[:, 1] * 30       # y: -30 to 30m
        lidar_points[:, 2] = lidar_points[:, 2] * 2 - 1    # z: -3 to 1m
        lidar_points[:, 3] = np.abs(lidar_points[:, 3])    # intensity
        
        # Generate radar points
        num_radar = np.random.randint(50, 200)
        radar_points = np.random.randn(num_radar, 5).astype(np.float32)
        
        # Generate camera image
        camera_image = np.random.rand(450, 800, 3).astype(np.float32)
        
        # Generate 3D bounding boxes
        num_boxes = np.random.randint(5, 15)
        gt_boxes = np.zeros((num_boxes, 9), dtype=np.float32)
        gt_boxes[:, 0] = np.random.uniform(10, 50, num_boxes)      # x
        gt_boxes[:, 1] = np.random.uniform(-20, 20, num_boxes)     # y
        gt_boxes[:, 2] = np.random.uniform(-1, 0, num_boxes)       # z
        gt_boxes[:, 3:6] = np.random.uniform(1, 4, (num_boxes, 3))  # w, l, h
        gt_boxes[:, 6] = np.random.uniform(-np.pi, np.pi, num_boxes)  # yaw
        gt_boxes[:, 7] = np.random.randint(0, 10, num_boxes)       # class
        gt_boxes[:, 8] = np.arange(num_boxes)                      # track_id
        
        dataset.append({
            'lidar_points': lidar_points,
            'radar_points': radar_points,
            'camera_image': camera_image,
            'gt_boxes_3d': gt_boxes,
        })
    
    return dataset

# Generate dataset
print('Generating simulated autonomous driving data...')
dataset = generate_autonomous_driving_data(num_samples=100)
print(f'✓ Generated {len(dataset)} samples')
print(f'  Each sample: LiDAR ({dataset[0]["lidar_points"].shape[0]} points), '
      f'{len(dataset[0]["gt_boxes_3d"])} objects')

## Step 2: Visualization Functions

In [ ]:
def visualize_bev(points, boxes=None, trajectory=None, title='Bird\'s Eye View'):
    """
    Visualize bird's eye view with point cloud, boxes, and trajectory.
    """
    fig, ax = plt.subplots(figsize=(12, 12))
    
    # Plot point cloud
    if points is not None and len(points) > 0:
        ax.scatter(points[:, 0], points[:, 1], s=0.5, c='gray', alpha=0.3, label='LiDAR')
    
    # Plot bounding boxes
    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            x, y, w, l, yaw = box[0], box[1], box[3], box[4], box[6]
            corners = get_box_corners_2d(box)
            rect = plt.Polygon(corners, fill=False, edgecolor='red', linewidth=2)
            ax.add_patch(rect)
            # Direction arrow
            dx, dy = np.cos(yaw) * 2, np.sin(yaw) * 2
            ax.arrow(x, y, dx, dy, head_width=0.5, fc='red', ec='red')
    
    # Plot trajectory
    if trajectory is not None and len(trajectory) > 0:
        ax.plot(trajectory[:, 0], trajectory[:, 1], 'g-', linewidth=3, label='Path')
        ax.scatter(trajectory[0, 0], trajectory[0, 1], s=100, c='green', marker='o', label='Start', zorder=5)
        ax.scatter(trajectory[-1, 0], trajectory[-1, 1], s=100, c='blue', marker='*', label='Goal', zorder=5)
    
    ax.set_xlim(0, 70)
    ax.set_ylim(-40, 40)
    ax.set_xlabel('X (m)', fontsize=12)
    ax.set_ylabel('Y (m)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    ax.legend()
    plt.tight_layout()
    return fig

def get_box_corners_2d(box):
    """Get 2D corners of rotated bounding box."""
    x, y, w, l, yaw = box[0], box[1], box[3], box[4], box[6]
    corners_local = np.array([[-w/2, -l/2], [w/2, -l/2], [w/2, l/2], [-w/2, l/2]])
    rot_mat = np.array([[np.cos(yaw), -np.sin(yaw)], [np.sin(yaw), np.cos(yaw)]])
    corners = (rot_mat @ corners_local.T).T + np.array([x, y])
    return corners

print('✓ Visualization functions ready')

## Step 3: PointPillars 3D Object Detection Model

In [ ]:
class PointPillars(nn.Module):
    """PointPillars: Fast 3D Object Detection from Point Clouds"""
    
    def __init__(self, num_classes=10):
        super().__init__()
        self.num_classes = num_classes
        
        # Pillar feature encoder
        self.pillar_encoder = nn.Sequential(
            nn.Linear(9, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        
        # 2D Backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.ReLU(),
        )
        
        # Detection head
        self.cls_head = nn.Conv2d(256, num_classes, 1)
        self.box_head = nn.Conv2d(256, 7, 1)
    
    def forward(self, points):
        # Simplified forward pass
        B = points.shape[0]
        # Create dummy output for demo
        cls_pred = torch.randn(B, self.num_classes, 32, 32)
        box_pred = torch.randn(B, 7, 32, 32)
        return {'cls_preds': cls_pred, 'box_preds': box_pred}

# Create model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = PointPillars(num_classes=10).to(device)
model.eval()

print(f'✓ PointPillars model created ({sum(p.numel() for p in model.parameters()):,} parameters)')

## Step 4: Multi-Object Tracking

In [ ]:
class KalmanFilter3D:
    """Kalman filter for 3D object tracking."""
    
    def __init__(self, initial_state, dt=0.1):
        self.dt = dt
        self.x = np.zeros(10)  # [x, y, z, w, l, h, yaw, vx, vy, vz]
        self.x[:7] = initial_state
        self.P = np.eye(10) * 10
    
    def predict(self):
        # Simple constant velocity model
        self.x[0] += self.x[7] * self.dt
        self.x[1] += self.x[8] * self.dt
        self.x[2] += self.x[9] * self.dt
        return self.x[:7]
    
    def update(self, measurement):
        self.x[:7] = 0.7 * self.x[:7] + 0.3 * measurement
        return self.x[:7]

class Track:
    """Single object track."""
    _next_id = 1
    
    def __init__(self, detection):
        self.id = Track._next_id
        Track._next_id += 1
        self.kf = KalmanFilter3D(detection)
        self.hits = 1
        self.age = 1
        self.time_since_update = 0
        self.state = detection
        self.history = [detection.copy()]
    
    def predict(self):
        self.state = self.kf.predict()
        self.age += 1
        self.time_since_update += 1
        return self.state
    
    def update(self, detection):
        self.state = self.kf.update(detection)
        self.hits += 1
        self.time_since_update = 0
        self.history.append(self.state.copy())
    
    def get_state(self):
        return self.state

class MultiObjectTracker:
    """Multi-object tracker using Kalman filter."""
    
    def __init__(self, max_age=3, min_hits=3, iou_threshold=0.3):
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.tracks = []
        self.frame_count = 0
    
    def update(self, detections):
        self.frame_count += 1
        
        # Predict
        for track in self.tracks:
            track.predict()
        
        # Match detections to tracks (simplified)
        matched, unmatched_dets = [], list(range(len(detections)))
        if len(self.tracks) > 0 and len(detections) > 0:
            # Simple nearest-neighbor matching
            for i, det in enumerate(detections):
                best_dist, best_track = float('inf'), None
                for track in self.tracks:
                    dist = np.linalg.norm(det[:2] - track.get_state()[:2])
                    if dist < best_dist:
                        best_dist, best_track = dist, track
                if best_dist < 5.0:  # threshold
                    best_track.update(det)
                    matched.append(i)
            unmatched_dets = [i for i in range(len(detections)) if i not in matched]
        
        # Create new tracks
        for i in unmatched_dets:
            self.tracks.append(Track(detections[i]))
        
        # Remove old tracks
        self.tracks = [t for t in self.tracks if t.time_since_update < self.max_age]
        
        # Return confirmed tracks
        return [t for t in self.tracks if t.hits >= self.min_hits]
    
    def reset(self):
        self.tracks = []
        self.frame_count = 0
        Track._next_id = 1

print('✓ Tracking system ready')

## Step 5: Path Planning

In [ ]:
class OccupancyGrid:
    """2D occupancy grid for path planning."""
    
    def __init__(self, resolution=0.2, width=100, height=100, origin=(0, -50)):
        self.resolution = resolution
        self.width = width
        self.height = height
        self.origin = origin
        self.grid_width = int(width / resolution)
        self.grid_height = int(height / resolution)
        self.grid = np.zeros((self.grid_height, self.grid_width), dtype=np.uint8)
    
    def update_from_boxes(self, boxes, safety_margin=2.0):
        self.grid.fill(0)
        for box in boxes:
            x, y, w, l = box[0], box[1], box[3] + safety_margin*2, box[4] + safety_margin*2
            x_min, x_max = int((x - w/2 - self.origin[0]) / self.resolution), int((x + w/2 - self.origin[0]) / self.resolution)
            y_min, y_max = int((y - l/2 - self.origin[1]) / self.resolution), int((y + l/2 - self.origin[1]) / self.resolution)
            x_min, x_max = max(0, x_min), min(self.grid_width-1, x_max)
            y_min, y_max = max(0, y_min), min(self.grid_height-1, y_max)
            self.grid[y_min:y_max+1, x_min:x_max+1] = 1
    
    def world_to_grid(self, x, y):
        gx = int((x - self.origin[0]) / self.resolution)
        gy = int((y - self.origin[1]) / self.resolution)
        if 0 <= gx < self.grid_width and 0 <= gy < self.grid_height:
            return (gx, gy)
        return None
    
    def grid_to_world(self, gx, gy):
        x = gx * self.resolution + self.origin[0] + self.resolution/2
        y = gy * self.resolution + self.origin[1] + self.resolution/2
        return (x, y)
    
    def is_occupied(self, gx, gy):
        if 0 <= gx < self.grid_width and 0 <= gy < self.grid_height:
            return self.grid[gy, gx] > 0
        return True

class AStarPlanner:
    """A* path planner."""
    
    def __init__(self, grid):
        self.grid = grid
        self.motions = [(0,1,1), (0,-1,1), (1,0,1), (-1,0,1),
                        (1,1,1.414), (1,-1,1.414), (-1,1,1.414), (-1,-1,1.414)]
    
    def plan(self, start, goal):
        start_grid = self.grid.world_to_grid(*start)
        goal_grid = self.grid.world_to_grid(*goal)
        if not start_grid or not goal_grid:
            return None
        
        # A* search
        open_set = [(0, start_grid)]
        came_from = {}
        g_score = {start_grid: 0}
        
        while open_set:
            _, current = heapq.heappop(open_set)
            
            if current == goal_grid:
                # Reconstruct path
                path = []
                while current in came_from:
                    path.append(self.grid.grid_to_world(*current))
                    current = came_from[current]
                path.append(start)
                return list(reversed(path))
            
            for dx, dy, cost in self.motions:
                neighbor = (current[0] + dx, current[1] + dy)
                if self.grid.is_occupied(*neighbor):
                    continue
                
                tentative_g = g_score[current] + cost
                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f_score = tentative_g + np.linalg.norm(np.array(neighbor) - np.array(goal_grid))
                    heapq.heappush(open_set, (f_score, neighbor))
        
        return None

class TrajectoryOptimizer:
    """Trajectory optimizer with velocity profile."""
    
    def __init__(self, max_velocity=15.0, max_acceleration=3.0, dt=0.1):
        self.max_velocity = max_velocity
        self.max_acceleration = max_acceleration
        self.dt = dt
    
    def optimize(self, waypoints, initial_velocity=5.0):
        if len(waypoints) < 2:
            return np.array(waypoints), np.array([initial_velocity]), np.array([0])
        
        waypoints = np.array(waypoints)
        
        # Smooth path using linear interpolation
        distances = np.cumsum([0] + [np.linalg.norm(waypoints[i+1] - waypoints[i]) for i in range(len(waypoints)-1)])
        num_points = max(int(distances[-1] / 0.5), 100)
        sample_distances = np.linspace(0, distances[-1], num_points)
        trajectory = np.array([
            np.interp(sample_distances, distances, waypoints[:, 0]),
            np.interp(sample_distances, distances, waypoints[:, 1]),
        ]).T
        
        # Compute velocity profile
        velocities = np.ones(len(trajectory)) * initial_velocity
        velocities = np.minimum(velocities, self.max_velocity)
        velocities[-1] = 0  # Stop at end
        
        timestamps = np.arange(len(trajectory)) * self.dt
        
        return trajectory, velocities, timestamps

print('✓ Path planning ready')

## Step 6: Run Complete Pipeline

Now let's run the complete perception system!

In [ ]:
# Visualize first sample
sample = dataset[0]
print('Sample 0:')
print(f'  LiDAR points: {len(sample["lidar_points"])}')
print(f'  Objects: {len(sample["gt_boxes_3d"])}')

fig = visualize_bev(
    points=sample['lidar_points'][:, :3],
    boxes=sample['gt_boxes_3d'][:, :7],
    title='Sample 0: LiDAR Point Cloud + 3D Bounding Boxes'
)
plt.show()

In [ ]:
# Run detection (using ground truth for demo)
print('\nRunning 3D object detection...')
detections = sample['gt_boxes_3d'][:, :7]
print(f'✓ Detected {len(detections)} objects')

# Run tracking
print('\nInitializing multi-object tracker...')
tracker = MultiObjectTracker(max_age=3, min_hits=3)

num_frames = min(10, len(dataset))
for i in range(num_frames):
    sample = dataset[i]
    detections = sample['gt_boxes_3d'][:, :7]
    tracks = tracker.update(detections)
    print(f'  Frame {i}: {len(detections)} detections → {len(tracks)} confirmed tracks')

print(f'✓ Tracking complete: {len(tracker.tracks)} total tracks')

In [ ]:
# Path planning
print('\nPlanning collision-free path...')
sample = dataset[num_frames-1]
tracks = tracker.update(sample['gt_boxes_3d'][:, :7])

if len(tracks) > 0:
    # Create occupancy grid
    grid = OccupancyGrid(resolution=0.2, width=100, height=100, origin=(0, -50))
    tracked_boxes = np.array([t.get_state() for t in tracks])
    grid.update_from_boxes(tracked_boxes, safety_margin=2.0)
    
    # Plan path
    planner = AStarPlanner(grid)
    start, goal = (5.0, 0.0), (50.0, 10.0)
    path = planner.plan(start, goal)
    
    if path:
        print(f'✓ Path found: {len(path)} waypoints')
        
        # Optimize trajectory
        optimizer = TrajectoryOptimizer(max_velocity=15.0, max_acceleration=3.0)
        trajectory, velocities, timestamps = optimizer.optimize(path, initial_velocity=5.0)
        print(f'✓ Trajectory optimized: {len(trajectory)} points, {timestamps[-1]:.1f}s duration')
        
        # Visualize result
        fig = visualize_bev(
            points=sample['lidar_points'][:, :3],
            boxes=tracked_boxes,
            trajectory=trajectory,
            title='Complete Pipeline: Detection + Tracking + Planning'
        )
        plt.show()
    else:
        print('⚠ No path found')
else:
    print('⚠ No tracks available')

In [ ]:
# Final summary
print('\n' + '='*80)
print('DEMO COMPLETE!')
print('='*80)
print('This notebook demonstrated:')
print('  ✓ Simulated multi-sensor data generation')
print('  ✓ 3D object detection with PointPillars')
print('  ✓ Multi-object tracking with Kalman filter')
print('  ✓ Path planning with A* algorithm')
print('  ✓ Trajectory optimization')
print('\nAll systems operational!')
print('='*80)